In [2]:
import os
import pandas as pd
import mlflow
import mlflow.sklearn

from mlflow import MlflowClient
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


# ======================================
# SETTINGS
# ======================================
TRACKING_URI = "http://127.0.0.1:5000"
EXPERIMENT_NAME = "Spotify_Churn_Logistic_Regression"
REGISTERED_MODEL_NAME = "spotify_churn_logreg_clean"
SET_CHAMPION_ALIAS = True


def build_pipeline(X_frame, selected_features, c_val):
    X_subset = X_frame[selected_features]

    numeric_features = X_subset.select_dtypes(
        include=["int64", "int32", "float64", "float32"]
    ).columns.tolist()

    categorical_features = X_subset.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)
        ]
    )

    model_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            C=c_val,
            max_iter=3000,
            random_state=42,
            class_weight="balanced",
            solver="liblinear",
            penalty="l2"
        ))
    ])

    return model_pipeline


def backward_selection(X_train, y_train, X_val, y_val, initial_features, c_val, min_features=3):
    selected_features = initial_features.copy()

    baseline_pipeline = build_pipeline(X_train, selected_features, c_val)
    baseline_pipeline.fit(X_train[selected_features], y_train)
    baseline_preds = baseline_pipeline.predict(X_val[selected_features])
    best_score = f1_score(y_val, baseline_preds, zero_division=0)

    improved = True

    while improved and len(selected_features) > min_features:
        improved = False
        feature_to_remove = None
        candidate_best_score = best_score

        for feature in selected_features:
            trial_features = [f for f in selected_features if f != feature]

            trial_pipeline = build_pipeline(X_train, trial_features, c_val)
            trial_pipeline.fit(X_train[trial_features], y_train)
            trial_preds = trial_pipeline.predict(X_val[trial_features])
            trial_score = f1_score(y_val, trial_preds, zero_division=0)

            if trial_score >= candidate_best_score:
                candidate_best_score = trial_score
                feature_to_remove = feature

        if feature_to_remove is not None:
            selected_features.remove(feature_to_remove)
            best_score = candidate_best_score
            improved = True

    return selected_features, best_score


def main():
    # -----------------------------
    # 1. SET UP MLFLOW
    # -----------------------------
    mlflow.set_tracking_uri(TRACKING_URI)
    mlflow.set_experiment(EXPERIMENT_NAME)

    print("Tracking URI:", TRACKING_URI)
    print("Experiment:", EXPERIMENT_NAME)
    print("Registered model name:", REGISTERED_MODEL_NAME)
    print()

    # -----------------------------
    # 2. LOAD DATA
    # -----------------------------
    file_name = "../data/processed/spotify_user_behavior_features.parquet"
    df = pd.read_parquet(file_name)

    print("Dataset loaded successfully.")
    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())
    print()

    # -----------------------------
    # 3. BASIC CLEANUP
    # -----------------------------
    target_column = "churn"

    if target_column not in df.columns:
        raise ValueError(f"Target column '{target_column}' not found in dataset.")

    if "signup_date" in df.columns:
        df["signup_date"] = pd.to_datetime(df["signup_date"], errors="coerce")
        df["days_since_signup_start"] = (
            df["signup_date"] - df["signup_date"].min()
        ).dt.days

    drop_columns = [
        "user_id",
        "signup_date",
        "inactive_3_months_flag",
        "months_inactive",
        "ad_conversion_to_subscription"
    ]

    df = df.dropna(subset=[target_column])

    X = df.drop(columns=drop_columns + [target_column], errors="ignore")
    y = df[target_column].astype(int)

    print("Dropped columns:")
    print(drop_columns)
    print()

    print("Final features available for backward selection:")
    print(X.columns.tolist())
    print()

    print("Target distribution:")
    print(y.value_counts())
    print()

    # -----------------------------
    # 4. TRAIN / TEST SPLIT
    # -----------------------------
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=0.25,
        random_state=42,
        stratify=y_train_full
    )

    initial_features = X.columns.tolist()

    # -----------------------------
    # 5. TRAIN MULTIPLE RUNS
    # -----------------------------
c_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

    best_f1 = -1.0
    best_c = None
    best_model = None
    best_selected_features = None
    best_removed_features = None
    best_selection_val_f1 = None
    best_accuracy = None
    best_precision = None
    best_recall = None
    best_roc_auc = None
    best_cm = None
    best_report = None

    for c_val in c_values:
        with mlflow.start_run(run_name=f"logreg_backward_balanced_no_leakage_C_{c_val}"):
            selected_features, selection_val_f1 = backward_selection(
                X_train,
                y_train,
                X_val,
                y_val,
                initial_features,
                c_val,
                min_features=3
            )

            removed_features = [f for f in initial_features if f not in selected_features]

            final_pipeline = build_pipeline(X_train_full, selected_features, c_val)
            final_pipeline.fit(X_train_full[selected_features], y_train_full)

            y_pred = final_pipeline.predict(X_test[selected_features])
            y_prob = final_pipeline.predict_proba(X_test[selected_features])[:, 1]

            print("=" * 55)
            print(f"Run for C = {c_val}")
            print("Predicted class counts:")
            print(pd.Series(y_pred).value_counts())
            print()
            print("Actual class counts:")
            print(pd.Series(y_test).value_counts())
            print()

            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall = recall_score(y_test, y_pred, zero_division=0)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            roc_auc = roc_auc_score(y_test, y_prob)
            cm = confusion_matrix(y_test, y_pred)
            report = classification_report(y_test, y_pred, zero_division=0)

            print("Selected features:")
            print(selected_features)
            print("Removed features:")
            print(removed_features)
            print("Selection validation F1:", selection_val_f1)
            print("Accuracy:", accuracy)
            print("Precision:", precision)
            print("Recall:", recall)
            print("F1 Score:", f1)
            print("ROC AUC:", roc_auc)
            print("Confusion Matrix:")
            print(cm)
            print("Classification Report:")
            print(report)

            # Log params
            mlflow.log_param("model_type", "LogisticRegression")
            mlflow.log_param("selection_method", "backward")
            mlflow.log_param("dataset_file", file_name)
            mlflow.log_param("target_column", target_column)
            mlflow.log_param("C", c_val)
            mlflow.log_param("max_iter", 3000)
            mlflow.log_param("class_weight", "balanced")
            mlflow.log_param("solver", "liblinear")
            mlflow.log_param("penalty", "l2")
            mlflow.log_param("train_rows", len(X_train_full))
            mlflow.log_param("test_rows", len(X_test))
            mlflow.log_param("selected_feature_count", len(selected_features))
            mlflow.log_param("removed_feature_count", len(removed_features))
            mlflow.log_param("selected_features", ",".join(selected_features))
            mlflow.log_param("removed_features", ",".join(removed_features))
            mlflow.log_param("dropped_columns", ",".join(drop_columns))
            mlflow.set_tag("model_type", "clean")
            mlflow.set_tag("leakage", "no")

            # Log metrics
            mlflow.log_metric("selection_validation_f1", selection_val_f1)
            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("f1_score", f1)
            mlflow.log_metric("roc_auc", roc_auc)

            # Log model artifact for experiment tracking
            safe_c_val = str(c_val).replace(".", "_")
            mlflow.sklearn.log_model(
                sk_model=final_pipeline,
                artifact_path="logreg_backward_balanced_no_leakage_model_C_" + safe_c_val
            )

            selected_file = f"selected_features_balanced_no_leakage_C_{safe_c_val}.csv"
            pd.DataFrame({"selected_feature": selected_features}).to_csv(selected_file, index=False)
            mlflow.log_artifact(selected_file)

            # Track best by F1
            if f1 > best_f1:
                best_f1 = f1
                best_c = c_val
                best_model = final_pipeline
                best_selected_features = selected_features
                best_removed_features = removed_features
                best_selection_val_f1 = selection_val_f1
                best_accuracy = accuracy
                best_precision = precision
                best_recall = recall
                best_roc_auc = roc_auc
                best_cm = cm
                best_report = report

    # -----------------------------
    # 6. REGISTER THE BEST CLEAN MODEL
    # -----------------------------
    print("=" * 55)
    print("Best C value:", best_c)
    print("Best F1 score:", best_f1)
    print("Registering best clean model...")
    print()

    with mlflow.start_run(run_name=f"logreg_clean_registered_C_{best_c}") as reg_run:
        mlflow.log_param("model_type", "LogisticRegression")
        mlflow.log_param("selection_method", "backward")
        mlflow.log_param("target_column", target_column)
        mlflow.log_param("dataset_file", file_name)
        mlflow.log_param("C", best_c)
        mlflow.log_param("max_iter", 3000)
        mlflow.log_param("class_weight", "balanced")
        mlflow.log_param("solver", "liblinear")
        mlflow.log_param("penalty", "l2")
        mlflow.log_param("selected_feature_count", len(best_selected_features))
        mlflow.log_param("removed_feature_count", len(best_removed_features))
        mlflow.log_param("selected_features", ",".join(best_selected_features))
        mlflow.log_param("removed_features", ",".join(best_removed_features))
        mlflow.log_param("dropped_columns", ",".join(drop_columns))
        mlflow.set_tag("model_type", "clean")
        mlflow.set_tag("leakage", "no")
        mlflow.set_tag("final_choice", "yes")

        mlflow.log_metric("selection_validation_f1", best_selection_val_f1)
        mlflow.log_metric("accuracy", best_accuracy)
        mlflow.log_metric("precision", best_precision)
        mlflow.log_metric("recall", best_recall)
        mlflow.log_metric("f1_score", best_f1)
        mlflow.log_metric("roc_auc", best_roc_auc)

        best_selected_file = "best_clean_selected_features.csv"
        pd.DataFrame({"selected_feature": best_selected_features}).to_csv(best_selected_file, index=False)
        mlflow.log_artifact(best_selected_file)

        mlflow.log_text(str(best_cm), "best_clean_confusion_matrix.txt")
        mlflow.log_text(best_report, "best_clean_classification_report.txt")

        mlflow.sklearn.log_model(
            sk_model=best_model,
            artifact_path="registered_clean_model",
            registered_model_name=REGISTERED_MODEL_NAME
        )

        print("Run ID:", reg_run.info.run_id)
        print("Registered model name:", REGISTERED_MODEL_NAME)

    # -----------------------------
    # 7. OPTIONAL: SET CHAMPION ALIAS
    # -----------------------------
    if SET_CHAMPION_ALIAS:
        client = MlflowClient(tracking_uri=TRACKING_URI)
        versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
        latest_version_number = max(int(v.version) for v in versions)

        client.set_registered_model_alias(
            name=REGISTERED_MODEL_NAME,
            alias="champion",
            version=latest_version_number
        )

        print("Latest registered version:", latest_version_number)
        print("Alias 'champion' assigned successfully.")
        print()

    print("Done.")
    print("Open MLflow UI at: http://127.0.0.1:5000")
    print(f"Check the Models tab for: {REGISTERED_MODEL_NAME}")


if __name__ == "__main__":
    main()


Tracking URI: http://127.0.0.1:5000
Experiment: Spotify_Churn_Logistic_Regression
Registered model name: spotify_churn_logreg_clean

Dataset loaded successfully.
Shape: (50000, 23)
Columns:
['user_id', 'country', 'age', 'signup_date', 'subscription_type', 'churn', 'months_inactive', 'inactive_3_months_flag', 'ad_interaction', 'ad_conversion_to_subscription', 'music_suggestion_rating_1_to_5', 'avg_listening_hours_per_week', 'favorite_genre', 'most_liked_feature', 'desired_future_feature', 'primary_device', 'playlists_created', 'avg_skips_per_day', 'likes_personalization', 'dislikes_suggestions', 'heavy_listener', 'many_playlists', 'new_user']

Dropped columns:
['user_id', 'signup_date', 'inactive_3_months_flag', 'months_inactive', 'ad_conversion_to_subscription']

Final features available for backward selection:
['country', 'age', 'subscription_type', 'ad_interaction', 'music_suggestion_rating_1_to_5', 'avg_listening_hours_per_week', 'favorite_genre', 'most_liked_feature', 'desired_futu

2026/04/16 00:13:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 00:13:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run logreg_backward_balanced_no_leakage_C_0.01 at: http://127.0.0.1:5000/#/experiments/1/runs/4aa9500b092a460e8e6434469cdd6919
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run for C = 0.1
Predicted class counts:
1    5061
0    4939
Name: count, dtype: int64

Actual class counts:
churn
0    8422
1    1578
Name: count, dtype: int64

Selected features:
['country', 'music_suggestion_rating_1_to_5', 'avg_listening_hours_per_week', 'primary_device', 'playlists_created', 'avg_skips_per_day', 'dislikes_suggestions', 'heavy_listener', 'many_playlists', 'new_user']
Removed features:
['age', 'subscription_type', 'ad_interaction', 'favorite_genre', 'most_liked_feature', 'desired_future_feature', 'likes_personalization', 'days_since_signup_start']
Selection validation F1: 0.25594405594405595
Accuracy: 0.5021
Precision: 0.16399920964236317
Recall: 0.5259822560202788
F1 Score: 0.2500376562735352
ROC AUC: 0.5045824969849321
Confusion Matrix:
[[4191 4231]
 [ 748  830]]
Classificat

2026/04/16 00:14:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 00:14:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run logreg_backward_balanced_no_leakage_C_0.1 at: http://127.0.0.1:5000/#/experiments/1/runs/f9e08e68178c4f98a38b58812bac8146
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run for C = 1.0
Predicted class counts:
0    5057
1    4943
Name: count, dtype: int64

Actual class counts:
churn
0    8422
1    1578
Name: count, dtype: int64

Selected features:
['country', 'subscription_type', 'ad_interaction', 'avg_listening_hours_per_week', 'favorite_genre', 'desired_future_feature', 'primary_device', 'playlists_created', 'likes_personalization', 'dislikes_suggestions', 'heavy_listener', 'many_playlists', 'days_since_signup_start']
Removed features:
['age', 'music_suggestion_rating_1_to_5', 'most_liked_feature', 'avg_skips_per_day', 'new_user']
Selection validation F1: 0.249212775528565
Accuracy: 0.5025
Precision: 0.15638276350394498
Recall: 0.48986058301647656
F1 Score: 0.23708020242294126
ROC AUC: 0.4989297900754226
Confusion Matrix:
[[4252 4170]
 [ 805  773]]
Classificati

...

🏃 View run logreg_backward_balanced_no_leakage_C_1.0 at: http://127.0.0.1:5000/#/experiments/1/runs/d239328e2a0b43c299d044effe0b4140
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run for C = 10.0
Predicted class counts:
0    5056
1    4944
Name: count, dtype: int64

Actual class counts:
churn
0    8422
1    1578
Name: count, dtype: int64

Selected features:
['country', 'subscription_type', 'ad_interaction', 'avg_listening_hours_per_week', 'favorite_genre', 'desired_future_feature', 'primary_device', 'playlists_created', 'likes_personalization', 'dislikes_suggestions', 'heavy_listener', 'many_playlists', 'days_since_signup_start']
Removed features:
['age', 'music_suggestion_rating_1_to_5', 'most_liked_feature', 'avg_skips_per_day', 'new_user']
Selection validation F1: 0.24917541229385307
Accuracy: 0.5026
Precision: 0.15655339805825244
Recall: 0.49049429657794674
F1 Score: 0.23735050597976082
ROC AUC: 0.4989251248841603
Confusion Matrix:
[[4252 4170]
 [ 804  774]]
Classific

2026/04/16 00:15:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 00:15:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run logreg_backward_balanced_no_leakage_C_10.0 at: http://127.0.0.1:5000/#/experiments/1/runs/165c01e39a8e41879c3688c7b27b4c85
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Best C value: 0.1
Best F1 score: 0.2500376562735352
Registering best clean model...



2026/04/16 00:15:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 00:15:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'spotify_churn_logreg_clean'.
2026/04/16 00:15:15 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: spotify_churn_logreg_clean, version 1
Created version '1' of model 'spotify_churn_logreg_clean'.


Run ID: 2a2a0290aad94b85865a92e5bb73789e
Registered model name: spotify_churn_logreg_clean
🏃 View run logreg_clean_registered_C_0.1 at: http://127.0.0.1:5000/#/experiments/1/runs/2a2a0290aad94b85865a92e5bb73789e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Latest registered version: 1
Alias 'champion' assigned successfully.

Done.
Open MLflow UI at: http://127.0.0.1:5000
Check the Models tab for: spotify_churn_logreg_clean
